In [1]:
import numpy as np
from matplotlib import colormaps
from matplotlib import pyplot as plt
from mne.io import read_raw_fif
from mne.time_frequency import psd_array_multitaper
from mne.preprocessing import (ICA, create_eog_epochs, create_ecg_epochs, corrmap)
from numpy.typing import NDArray
from scipy.integrate import simpson
from scipy.signal import periodogram, welch


from mne_lsl.player import PlayerLSL
from mne_lsl.stream import StreamLSL

from mne_lsl.lsl import (
    StreamInfo,
    StreamInlet,
    StreamOutlet,
    local_clock,
    resolve_streams,
)

from asrpy import asr_calibrate, asr_process, clean_windows
from hypyp import analyses

import pathlib

import numpy as np
import websockets
import asyncio

import time
# import cv2



c:\Users\thiag\anaconda3\Lib\site-packages\paramiko\pkey.py:82: CryptographyDeprecationWarning: TripleDES has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.TripleDES and will be removed from this module in 48.0.0.
  "cipher": algorithms.TripleDES,
c:\Users\thiag\anaconda3\Lib\site-packages\paramiko\transport.py:219: CryptographyDeprecationWarning: Blowfish has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.Blowfish and will be removed from this module in 45.0.0.
  "class": algorithms.Blowfish,
c:\Users\thiag\anaconda3\Lib\site-packages\paramiko\transport.py:243: CryptographyDeprecationWarning: TripleDES has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.TripleDES and will be removed from this module in 48.0.0.
  "class": algorithms.TripleDES,


In [2]:
nchan = 16
sfreq = 500 # Hz
bufferSize = 6 # seconds
nsamples = sfreq*bufferSize
LeadBuffer = np.zeros([nchan,nsamples])
FollowBuffer = np.zeros([nchan,nsamples])
connCohBuffer = np.zeros([nchan*2,nchan*2])


this_path = pathlib.Path().absolute()


In [3]:
this_path

WindowsPath('d:/Documents/Git_Repos/Real-time_EEG-Hyperscanning')

# Mocking stream from recorded .fif

In [5]:

fname_lead = this_path / "simple1_lead_eeg.fif"
player_lead = PlayerLSL(fname_lead, chunk_size=256, name = 'lead', n_repeat=10).start()
player_lead.info

fname_follow = this_path / "simple1_follow_eeg.fif"
player_follow = PlayerLSL(fname_follow, chunk_size=256, name = 'follow', n_repeat=10).start()
player_follow.info

Opening raw data file d:\Documents\Git_Repos\Real-time_EEG-Hyperscanning\simple1_lead_eeg.fif...
    Range : 35625 ... 127753 =     71.250 ...   255.506 secs
Ready.
Reading 0 ... 92128  =      0.000 ...   184.256 secs...
Opening raw data file d:\Documents\Git_Repos\Real-time_EEG-Hyperscanning\simple1_follow_eeg.fif...
    Range : 35685 ... 127813 =     71.370 ...   255.626 secs
Ready.
Reading 0 ... 92128  =      0.000 ...   184.256 secs...


Measurement date,Unknown
Experimenter,Unknown
Participant,Unknown
Digitized points,19 points
Good channels,"16 EEG, 3 Stimulus"
Bad channels,None
EOG channels,Not available
ECG channels,Not available
Sampling frequency,500.00 Hz
Highpass,0.50 Hz
Lowpass,35.00 Hz


# Getting the stream

In [6]:
streams = resolve_streams();

print(streams);


[< sInfo 'lead' >
  | Sampling: 500.0 Hz
  | Number of channels: 19
  | Data type: <class 'numpy.float64'>
  | Source: MNE-LSL
, < sInfo 'follow' >
  | Sampling: 500.0 Hz
  | Number of channels: 19
  | Data type: <class 'numpy.float64'>
  | Source: MNE-LSL
]


In [7]:
stream_lead = StreamLSL(bufsize=1, name="lead").connect()
stream_follow = StreamLSL(bufsize=1, name="follow").connect()

In [9]:
stream_lead.pick("eeg")  # channel selection
#assert "CPz" not in stream_lead.ch_names  # reference absent from the data stream
#stream_lead.add_reference_channels("CPz")

# stream_lead.set_eeg_reference("average")
stream_lead.info

stream_follow.pick("eeg")  # channel selection
#assert "CPz" not in stream_follow.ch_names  # reference absent from the data stream
#stream_follow.add_reference_channels("CPz")
# stream_follow.set_eeg_reference("average")
stream_follow.info



Measurement date,Unknown
Experimenter,Unknown
Participant,Unknown
Digitized points,19 points
Good channels,16 EEG
Bad channels,None
EOG channels,Not available
ECG channels,Not available
Sampling frequency,500.00 Hz
Highpass,0.50 Hz
Lowpass,35.00 Hz


In [8]:
freq_bands = {'Alpha-mu':[8,12]}

In [10]:
# close the player streams since we will be using the stream objects to read the data
player_lead.stop()
player_follow.stop()


<Player: follow | OFF | d:\Documents\Git_Repos\Real-time_EEG-Hyperscanning\simple1_follow_eeg.fif>

In [ ]:
# create a buffer to hold 1 minute of data for each stream
buffer_size = int(60 * stream_lead.info["sfreq"])
asr_template_lead = np.zeros((buffer_size, len(stream_lead.ch_names)))
buffer_follow = np.zeros((buffer_size, len(stream_follow.ch_names)))

In [ ]:
from asrpy import ASR

asr_lead = ASR(sfreq=stream_lead.info["sfreq"], cutoff=20)
asr_lead.fit(stream_lead.filter(l_freq=1, h_freq=30.))

asr_follow = ASR(sfreq=stream_follow.info["sfreq"], cutoff=20)
asr_follow.fit(stream_follow.filter(l_freq=1, h_freq=30.))



AttributeError: 'StreamLSL' object has no attribute 'copy'

In [ ]:
stream_lead_clean = asr_lead.transform(stream_lead.filter(l_freq=0.5, h_freq=45.))
stream_follow_clean = asr_follow.transform(stream_follow.filter(l_freq=0.5, h_freq=45.))

In [9]:
 # Update and send the array
t = time.time()
# l = stream_lead.filter(l_freq=0.5, h_freq=30.).get_data(picks='eeg')[0]
# f = stream_follow.filter(l_freq=0.5, h_freq=30.).get_data(picks='eeg')[0]



streamLength = l.shape[1]

LeadBuffer = np.concatenate((LeadBuffer[:,streamLength:],l), axis=1)
FollowBuffer = np.concatenate((FollowBuffer[:,streamLength:],f), axis=1)

clean_l = asr_process(LeadBuffer, sfreq, M_l, T_l)
clean_f = asr_process(FollowBuffer, sfreq, M_f, T_f)

epoch = np.stack((clean_l[:,streamLength:], clean_f[:,streamLength:]))
epoch_reshape = epoch.reshape(2, 1, nchan, nsamples-streamLength)

complex_epochs = analyses.compute_freq_bands(epoch_reshape, freq_bands=freq_bands, sampling_rate=sfreq)
conn_coh = analyses.compute_sync(complex_epochs, mode='coh', epochs_average=True)
print(conn_coh.shape)
nconn_coh = conn_coh[0, 0:nchan, nchan:2*nchan]
print(nconn_coh)

print(time.time() - t)
# print(nconn_coh)

plt.plot(clean_l.T[2000:]);
plt.show();
plt.plot(l.T);
plt.show();



NameError: name 'l' is not defined

In [97]:
print(conn_coh.shape)
print(nconn_coh.shape)

(1, 1, 32, 32)
(16, 16)


In [98]:
async def send_array(websocket, stop_event):
    while not stop_event.is_set():
        # Update and send the array
        l = stream_lead.get_data()[0]
        clean_l = asr_process(l, sfreq, M_l, T_l)
        f = stream_follow.get_data()[0]
        clean_f = asr_process(f, sfreq, M_f, T_f)

        epoch = np.stack((clean_l, clean_f))
        print(epoch.shape)
        epoch_reshape = epoch.reshape(2, 1, 16, 500)

        complex_epochs = analyses.compute_freq_bands(epoch_reshape, freq_bands=freq_bands, sampling_rate=sfreq)
        conn_coh = analyses.compute_sync(complex_epochs, mode='coh', epochs_average=False)
        conn_coh = conn_coh[0, 0, :, :]

        array_str = '\n'.join([','.join(map(str, row)) for row in conn_coh])
        await websocket.send(array_str)

        # Wait for 500ms before sending the next update
        await asyncio.sleep(0.5)

async def main():
    stop_event = asyncio.Event()
    async with websockets.serve(lambda ws, path: send_array(ws, stop_event), "localhost", 8080) as server:
        await stop_event.wait()
        await server.wait_closed()

loop = asyncio.get_event_loop()
try:
    loop.run_until_complete(main())
finally:
    loop.close()

RuntimeError: Cannot close a running event loop

In [111]:
stop_event.set()
loop.stop()

NameError: name 'stop_event' is not defined

In [113]:
stream_lead.disconnect()
stream_follow.disconnect()

RuntimeError: The Stream is not connected. Please connect to the stream with the method stream.connect(...) to use StreamLSL.disconnect().

In [114]:
player_lead.stop()
# player_follow.stop()

RuntimeError: The player is not started. Use Player.start() to begin streaming.